In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['CRYPTOGRAPHY_OPENSSL_NO_LEGACY'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

import datasets
from datasets import load_dataset, Dataset, DatasetDict, Dataset
from tqdm.auto import tqdm
import json

datasets.disable_caching()

/mnt/nas/home/genggui/miniconda3/envs/megatron-next/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from megatron.energon import get_train_dataset, WorkerConfig, get_loader, DefaultTaskEncoder, Cooker, basic_sample_keys
from megatron.energon import DefaultTaskEncoder, TextSample, Sample
import gzip
import pickle
import webdataset as wds
import json
import dataclasses
import torch

from tqdm.auto import tqdm
from typing import List

from qwen3_base_energon_helpers import MyTaskEncoder, print_error_handler

In [3]:
tokenizer_path="/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_80b_a3b_next_gemini_bf16/tokenizer_v19_gemini"

In [4]:
worker_config = WorkerConfig(
    rank=0,
    world_size=1,
    seed_offset=123456789,
    num_workers=1,
)

train_ds = get_train_dataset(
    '/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_80b_a3b_next_gemini_bf16/dataset.yaml',
    split_part="train",
    task_encoder=MyTaskEncoder(
        tokenizer_path=tokenizer_path,
        seq_length=66536,
        sensitive_words_path=None,
    ),
    batch_size=1,
    packing_buffer_size=None,
    shuffle_buffer_size=None,
    max_samples_per_sequence=None,
    worker_config=worker_config,
)

train_dataloader = get_loader(train_ds, worker_config=worker_config,)

rank=0, worker=0: sample_range=[0, 98994] in 1 slices, sum(count)=98994: indexes=[0, 98994] slices=[train-chunk-0.tar[0(start), 98994(end)]]
rank=0, worker=0: sample_range=[0, 33580] in 1 slices, sum(count)=33580: indexes=[0, 33580] slices=[train-chunk-0.tar[0(start), 33580(end)]]
rank=0, worker=0: sample_range=[0, 70519] in 1 slices, sum(count)=70519: indexes=[0, 70519] slices=[train-chunk-0.tar[0(start), 70519(end)]]
rank=0, worker=0: sample_range=[0, 99256] in 1 slices, sum(count)=99256: indexes=[0, 99256] slices=[train-chunk-0.tar[0(start), 99256(end)]]
rank=0, worker=0: sample_range=[0, 14160] in 1 slices, sum(count)=14160: indexes=[0, 14160] slices=[train-chunk-0.tar[0(start), 14160(end)]]
rank=0, worker=0: sample_range=[0, 5231] in 1 slices, sum(count)=5231: indexes=[0, 5231] slices=[train-chunk-0.tar[0(start), 5231(end)]]
rank=0, worker=0: sample_range=[0, 1453] in 1 slices, sum(count)=1453: indexes=[0, 1453] slices=[train-chunk-0.tar[0(start), 1453(end)]]
rank=0, worker=0: sam

/mnt/nas/home/genggui/miniconda3/envs/megatron-next/lib/python3.12/site-packages/megatron/energon/loader.py:108: FutureWarning: Passing a worker_config to get_loader() is deprecated and will have no effect.
  warn_deprecated(


In [5]:
dd = iter(train_dataloader)

Token indices sequence length is longer than the specified maximum sequence length for this model (116641 > 66536). Running this sequence through the model will result in indexing errors


In [6]:
import json

count_map = {}

with open("/mnt/nas/home/genggui/code/Megatron-Next/model_dir/pulse_v19_2_80b_a3b_next_gemini_bf16/qua/pulse_v19_32b_gemini_train.jsonl", "w", encoding="utf8") as f:
    for _ in tqdm(range(2048)):
        item = next(dd)
        
        kk = item['__key__'][0].split("/")[-1].split("-train-")[0]
        if kk not in count_map:
            count_map[kk] = 0
        
        count_map[kk] += 1
              
        item = {
            "input_ids": item['input_ids'].tolist()[0],
            "labels": item['labels'].tolist()[0],
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        

            
        

100%|██████████| 2048/2048 [00:17<00:00, 116.98it/s]


In [7]:
count_list = sorted(count_map.items(), key=lambda x:-x[1])
[(k,v/2048)for k,v in count_list]

[('gg_en_fineweb-edu_data', 0.07568359375),
 ('gg_zh_gg_zh_edu_data', 0.06982421875),
 ('zh_med_our_med_kb', 0.05419921875),
 ('gg_zh_med_ns_data', 0.04150390625),
 ('evol-codealpaca-v1', 0.0341796875),
 ('exam-math-cn', 0.02392578125),
 ('WildChat', 0.02294921875),
 ('预问诊', 0.0224609375),
 ('exam-en', 0.02001953125),
 ('aya_dataset', 0.01953125),
 ('multi_my_function_tools', 0.01904296875),
 ('gg_en_med_ns_data', 0.017578125),
 ('function_call', 0.01708984375),
 ('general-cn-my', 0.0166015625),
 ('rpg_zh', 0.0146484375),
 ('general_cn_base-same-language-think', 0.01416015625),
 ('NuminaMath-TIR', 0.01416015625),
 ('exam-science-cn', 0.01318359375),
 ('tulu-3-IF-augmented', 0.01318359375),
 ('general_multilingual_base-en-think', 0.01318359375),
 ('DeepMath-103K-same-language-think', 0.01171875),
 ('zh_ChinaNews-cn_data', 0.01171875),
 ('roleplay-zh-sharegpt-gpt4-data', 0.01171875),
 ('freedomIntelligence_general_en', 0.01123046875),
 ('tulu-3-sft-personas-code', 0.01123046875),
 ('long

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

In [9]:
tokenizer("<|im_start|>assistant\n<think>\n").input_ids 

[151644, 77091, 198, 13708, 766, 397]

In [10]:
tokenizer("<|im_start|>assistant\n<ggthink>\n").input_ids 

[151644, 77091, 198, 151667, 198]

In [14]:
item = next(dd)
# print(item['__key__'][0].split("/")[-1].split("-train-")[0])
print(tokenizer.decode([item for item in item['input_ids'].tolist()[0] if item != -100]))

<|im_start|>system
Use English in your thinking process.<|im_end|>
<|im_start|>user
患女，45岁。以“多颗牙齿脱落，咀嚼困难10余年”为主诉就诊。患者自述约14年前开始出现双手胀感及晨僵，伴有双手指关节间歇性疼痛。随后数年，逐渐感到面部皮肤发紧、干燥，并自觉口干、眼干，需频繁饮水。在此期间，出现进行性牙龈萎缩、牙根暴露，多数牙齿相继自行脱落或因重度松动而被拔除。专科检查：面部皮肤粗糙干燥、弹性差，鼻部小而陡峭，口裂显著变小，口唇紧绷，表情淡漠。面下三分之一垂直距离缩短并凹陷。口腔检查示17～26，37～46牙缺失，下颌牙槽骨严重吸收，牙槽嵴低平呈刃状。开口度约3cm，可见唾液分泌显著减少，质地粘稠。颞下颌关节活动正常但患者自述开闭口时偶有弹响。

根据患者信息，给出患者的主诊断和三个鉴别诊断，并对每个鉴别诊断给出简洁的理由说明，鉴别诊断需要按照可能性从高到低排序，用json格式返回，格式如下：
```json
[
    {
        "主诊断": "aaaa", 
        "理由: "xxxxx", 
    },
    {
        "鉴别诊断": "aaaa", 
        "理由: "xxxxx", 
    },
    {
        "鉴别诊断": "bbb", 
        "理由: "xxxxx", 
    },
    {
        "鉴别诊断": "ccc", 
        "理由: "xxxxx", 
    },
]
```<|im_end|>
<|im_start|>assistant
<ggthink>
Here's a thinking process that leads to the desired JSON output:

1.  **Deconstruct the User's Request:**
    *   **Input:** A detailed medical case report of a 45-year-old female patient.
    *   **Task:**
        *   Provide a primary diagnosis (主诊断).
  

In [12]:
print(tokenizer.decode([item for item in item['labels'].tolist()[0] if item != -100]))

Okay, here is a list of well-regarded organizations and communities that offer support for individuals exploring or struggling with issues related to sexual orientation and gender identity. These groups are known for providing safe and affirming resources:

1.  **The Trevor Project:** Focused on crisis intervention and suicide prevention for LGBTQ young people under 25. They offer:
    *   **TrevorLifeline:** 24/7 phone support.
    *   **TrevorText & TrevorChat:** Confidential text and online instant messaging with counselors.
    *   **TrevorSpace:** An affirming international social networking site for LGBTQ youth ages 13-24.
    *   Website: [https://www.thetrevorproject.org/](https://www.thetrevorproject.org/)

2.  **PFLAG (Parents, Families, and Friends of Lesbians and Gays):** The first and largest organization for LGBTQ+ people, their parents and families, and allies. They offer:
    *   Peer support meetings (often through local chapters).
    *   Online resources and publicat